In [18]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd

## Clean & Merge Data EMS & ACS Data 

In [19]:
ems = pd.read_parquet("../data/processed/nyc_ems_2024q1_usable.parquet")
acs = pd.read_parquet("../data/raw/acs_2022_acs5_zcta_full.parquet")

nyc_zips = ems["zipcode"].dropna().unique()

acs_nyc = acs[acs["zipcode"].isin(nyc_zips)].copy()

print("Total ACS ZIPs:", len(acs))
print("NYC ZIPs matched:", len(acs_nyc))

Total ACS ZIPs: 33774
NYC ZIPs matched: 207


In [20]:
# clean ACS median income (remove suppressed/invalid)
acs_nyc["median_income"] = acs_nyc["median_income"].where(acs_nyc["median_income"] > 0, pd.NA)
print(acs_nyc["median_income"].describe())

count       183.000000
mean      93973.617486
std       43642.469164
min       26400.000000
25%       64625.500000
50%       84691.000000
75%      112318.000000
max      250001.000000
Name: median_income, dtype: float64


In [21]:
#save NYC-only ACS to data/raw 
os.makedirs("../data/raw", exist_ok=True)
acs_nyc_out = "../data/raw/acs_2022_zip.parquet"
acs_nyc.to_parquet(acs_nyc_out, index=False)
print("saved:", acs_nyc_out)

saved: ../data/raw/acs_2022_zip.parquet


In [22]:
ems = pd.read_parquet("../data/processed/nyc_ems_2024q1_usable.parquet")
acs = pd.read_parquet("../data/raw/acs_2022_zip.parquet")

merged = ems.merge(acs, on ="zipcode", how="left")

print("EMS rows:", len(ems))
print("Merged rows:", len(merged))
print("Missing demographic rows:", merged["median_income"].isna().sum())

merged.head()

EMS rows: 363483
Merged rows: 363483
Missing demographic rows: 4369


,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,valid_dispatch_rspns_time_indc,dispatch_response_seconds_qy,valid_incident_rspns_time_indc,incident_response_seconds_qy,...,borough,zipcode,median_income,total_pop,white_pop,black_pop,hispanic_pop,pct_black,pct_white,pct_hispanic
0,240010001,2024-01-01 00:00:03,INJURY,5,INJURY,5,Y,53,Y,393.0,...,QUEENS,11365,82655.0,46424.0,12545.0,5461.0,9859.0,0.117633,0.270227,0.212369
1,240010002,2024-01-01 00:00:14,CARD,3,CARD,3,Y,84,Y,430.0,...,BRONX,10457,41145.0,79817.0,11647.0,34138.0,52152.0,0.427703,0.145921,0.653395
2,240010004,2024-01-01 00:00:45,RESPIR,4,RESPIR,4,Y,1485,Y,1934.0,...,MANHATTAN,10033,77209.0,58711.0,19188.0,4198.0,39448.0,0.071503,0.326821,0.671901
3,240010005,2024-01-01 00:00:55,BURNMI,4,BURNMI,4,Y,39,Y,1315.0,...,BROOKLYN,11208,56298.0,108180.0,18676.0,52901.0,42173.0,0.489009,0.172638,0.389841
4,240010006,2024-01-01 00:01:12,EDPC,7,EDPC,7,Y,11,Y,1050.0,...,RICHMOND / STATEN ISLAND,10309,110123.0,34740.0,30814.0,171.0,3322.0,0.004922,0.886989,0.095625


In [23]:
os.makedirs("../data/processed", exist_ok=True)

merged_out = "../data/processed/merged_ems_acs.parquet"
merged.to_parquet(merged_out, index=False)
print("saved:", merged_out)

saved: ../data/processed/merged_ems_acs.parquet


In [ ]:
#grouping vars
merged["income_q"] = pd.qcut(
    merged["median_income"],
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
)

median_black = merged["pct_black"].median()
merged["high_black_zip"] = (merged["pct_black"] > median_black).astype(int)

merged[["median_income", "income_q", "pct_black", "high_black_zip"]].head()

#save merged with groups to data/processed for severity grouping notebook
merged_w_groups_out = "../data/processed/merged_ems_acs_w_groups.parquet"
merged.to_parquet(merged_w_groups_out, index=False)
print("saved:", merged_w_groups_out)

saved: ../data/processed/merged_ems_acs_w_groups.parquet
